# 04 — Exploratory Analysis: Quarterfinal Teams vs. Eliminated Teams

**Prerequisite:** Run Notebooks 01 and 02 first to generate `data/processed/match_features.csv` and `data/processed/team_features.csv`.

**Goal:** Answer five focused questions about passing network structure using summary tables, distribution plots, boxplots, and a correlation matrix.

1. Do quarterfinal teams have **higher network density**?
2. Do quarterfinal teams have **lower degree centralization**?
3. Do quarterfinal teams have **more unique passing pairs**?
4. Do quarterfinal teams have **higher progressive pass share**?
5. Are successful teams more **balanced** or more reliant on a **single hub**?

---

**Two levels of analysis:**
- **Match-level** (n ≈ 128 team-match rows) — treats every match independently
- **Team-level** (n = 32) — aggregated across all a team's matches; cleaner signal

**Statistical note:** n=32 teams is small.  We use Mann-Whitney U (non-parametric) and report effect sizes (rank-biserial correlation r).  A result can be practically meaningful even if p > 0.05 when n is this small.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy import stats
from scipy.stats import mannwhitneyu

# ── visual style ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
})

QF_COLOR   = '#1a6faf'   # blue  — quarterfinalists
ELIM_COLOR = '#c0392b'   # red   — eliminated
NEUTRAL    = '#7f8c8d'

FIG_DIR = Path('../outputs/figures')
TAB_DIR = Path('../outputs/tables')
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── load feature tables ──────────────────────────────────────────────────────
mf = pd.read_csv('../data/processed/match_features.csv')
tf = pd.read_csv('../data/processed/team_features.csv')

QF_TEAMS = tf[tf['reached_quarterfinal'] == 1]['team'].tolist()

print(f"Match-level rows : {len(mf):>4}  "
      f"(QF team-matches: {(mf['reached_quarterfinal']==1).sum()}, "
      f"Elim: {(mf['reached_quarterfinal']==0).sum()})")
print(f"Team-level rows  : {len(tf):>4}  "
      f"(QF: {(tf['reached_quarterfinal']==1).sum()}, "
      f"Elim: {(tf['reached_quarterfinal']==0).sum()})")
print(f"\nQuarterfinalists : {', '.join(sorted(QF_TEAMS))}")

---
## Section 1 — Summary Statistics Table

Mean, median, and standard deviation for every network feature, split by quarterfinal advancement.

In [ ]:
# Features to analyse (match-level names, no _mean suffix)
FEATURES = [
    'network_density',
    'degree_centralization',
    'top_player_pass_share',
    'unique_passing_pairs',
    'total_completed_passes',
    'average_weighted_degree',
    'clustering_coefficient',
    'mean_betweenness_centrality',
    'max_betweenness_centrality',
    'progressive_pass_share',
    'final_third_entry_share',
    'centre_channel_share',
    'lateral_imbalance',
    'number_of_players_in_network',
]
FEATURES = [f for f in FEATURES if f in mf.columns]

LABELS = {
    'network_density'            : 'Network Density',
    'degree_centralization'      : 'Degree Centralization',
    'top_player_pass_share'      : 'Top Player Pass Share',
    'unique_passing_pairs'       : 'Unique Passing Pairs',
    'total_completed_passes'     : 'Total Completed Passes',
    'average_weighted_degree'    : 'Avg Weighted Degree',
    'clustering_coefficient'     : 'Clustering Coefficient',
    'mean_betweenness_centrality': 'Mean Betweenness',
    'max_betweenness_centrality' : 'Max Betweenness',
    'progressive_pass_share'     : 'Progressive Pass Share',
    'final_third_entry_share'    : 'Final Third Entry Share',
    'centre_channel_share'       : 'Centre Channel Share',
    'lateral_imbalance'          : 'Lateral Imbalance',
    'number_of_players_in_network': '# Players in Network',
}

In [ ]:
def mwu_row(feat, df, group_col='reached_quarterfinal'):
    """Return Mann-Whitney U p-value and rank-biserial effect size."""
    g1 = df[df[group_col] == 1][feat].dropna()
    g0 = df[df[group_col] == 0][feat].dropna()
    if len(g1) < 2 or len(g0) < 2:
        return np.nan, np.nan
    stat, p = mannwhitneyu(g1, g0, alternative='two-sided')
    rbc = 1 - (2 * stat) / (len(g1) * len(g0))
    return round(p, 4), round(rbc, 3)


rows = []
for feat in FEATURES:
    g1 = mf[mf['reached_quarterfinal'] == 1][feat].dropna()
    g0 = mf[mf['reached_quarterfinal'] == 0][feat].dropna()
    p, rbc = mwu_row(feat, mf)
    rows.append({
        'Feature'        : LABELS.get(feat, feat),
        'QF+ mean'       : g1.mean(),
        'QF+ median'     : g1.median(),
        'QF+ std'        : g1.std(),
        'Elim mean'      : g0.mean(),
        'Elim median'    : g0.median(),
        'Elim std'       : g0.std(),
        'Δ mean (QF−El)' : g1.mean() - g0.mean(),
        'Effect size (r)': rbc,
        'p-value'        : p,
        'Sig'            : '✓' if (p is not np.nan and p < 0.05) else '',
    })

summary = pd.DataFrame(rows).set_index('Feature')
summary.to_csv(TAB_DIR / '04_summary_stats.csv')

# Display rounded
fmt = {c: '{:.4f}' for c in summary.select_dtypes('float').columns}
summary.style.format(fmt).background_gradient(
    subset=['Effect size (r)'], cmap='RdBu', vmin=-0.4, vmax=0.4
)

---
## Section 2 — Boxplots: All Key Features Side by Side

Each subplot shows the distribution of one feature for QF teams (blue) vs. eliminated teams (red).
Dots = individual team-match observations; the line inside the box = median.

In [ ]:
ncols = 4
nrows = int(np.ceil(len(FEATURES) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.6))
axes = axes.flatten()

rng = np.random.default_rng(42)

for i, feat in enumerate(FEATURES):
    ax = axes[i]
    g1 = mf[mf['reached_quarterfinal'] == 1][feat].dropna().values
    g0 = mf[mf['reached_quarterfinal'] == 0][feat].dropna().values

    for pos, vals, color in [(0, g0, ELIM_COLOR), (1, g1, QF_COLOR)]:
        bp = ax.boxplot(
            vals, positions=[pos], widths=0.38,
            patch_artist=True,
            boxprops=dict(facecolor=color, alpha=0.45, linewidth=1.2),
            medianprops=dict(color='white', linewidth=2.5),
            whiskerprops=dict(color=color, linewidth=1.2),
            capprops=dict(color=color, linewidth=1.2),
            flierprops=dict(marker='', alpha=0),
        )
        jit = rng.uniform(-0.12, 0.12, size=len(vals))
        ax.scatter(pos + jit, vals, color=color, alpha=0.4, s=14, zorder=3)

    p, rbc = mwu_row(feat, mf)
    sig_str = f'p={p:.3f}' + (' *' if p < 0.05 else '')
    rbc_str = f'r={rbc:+.2f}' if rbc is not np.nan else ''
    ax.set_title(f"{LABELS.get(feat, feat)}\n{rbc_str}  {sig_str}",
                 fontsize=8.5, pad=4)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Elim', 'QF+'], fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3g'))
    ax.set_facecolor('#f9f9f9')

for j in range(len(FEATURES), len(axes)):
    axes[j].set_visible(False)

legend_handles = [
    mpatches.Patch(facecolor=QF_COLOR,   alpha=0.7, label='QF+ (reached quarterfinal)'),
    mpatches.Patch(facecolor=ELIM_COLOR, alpha=0.7, label='Eliminated'),
]
fig.legend(handles=legend_handles, loc='lower right',
           fontsize=9, framealpha=0.9)

fig.suptitle('Passing Network Features: Quarterfinal Teams vs. Eliminated Teams\n'
             '(match-level observations; r = rank-biserial effect size)',
             fontsize=12, y=1.01)
fig.tight_layout()
plt.savefig(FIG_DIR / '04_boxplots_all_features.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3 — Distribution Plots (KDE Overlays)

Kernel density estimates let us see the full shape of each distribution — not just the median and spread.  
The vertical dashed lines mark the group means.

In [ ]:
from scipy.stats import gaussian_kde

FOCAL_FEATURES = [
    'network_density',
    'degree_centralization',
    'top_player_pass_share',
    'unique_passing_pairs',
    'progressive_pass_share',
    'clustering_coefficient',
]
FOCAL_FEATURES = [f for f in FOCAL_FEATURES if f in mf.columns]

ncols = 3
nrows = int(np.ceil(len(FOCAL_FEATURES) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.5))
axes = axes.flatten()

for i, feat in enumerate(FOCAL_FEATURES):
    ax = axes[i]
    for group_val, color, label in [
        (0, ELIM_COLOR, 'Eliminated'),
        (1, QF_COLOR,   'QF+'),
    ]:
        vals = mf[mf['reached_quarterfinal'] == group_val][feat].dropna().values
        if len(vals) < 4:
            continue
        x_grid = np.linspace(vals.min() - vals.std(), vals.max() + vals.std(), 300)
        kde    = gaussian_kde(vals, bw_method='scott')
        density = kde(x_grid)

        ax.fill_between(x_grid, density, alpha=0.25, color=color)
        ax.plot(x_grid, density, color=color, lw=2, label=label)
        ax.axvline(vals.mean(), color=color, lw=1.4,
                   linestyle='--', alpha=0.85)

    p, rbc = mwu_row(feat, mf)
    sig = ' *' if p < 0.05 else ''
    ax.set_title(f"{LABELS.get(feat, feat)}\nr={rbc:+.2f}  p={p:.3f}{sig}",
                 fontsize=9)
    ax.set_xlabel(feat.replace('_', ' '), fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    ax.legend(fontsize=8)
    ax.set_facecolor('#f9f9f9')

for j in range(len(FOCAL_FEATURES), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribution Plots — Key Network Features (match level)',
             fontsize=12, y=1.01)
fig.tight_layout()
plt.savefig(FIG_DIR / '04_kde_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 4 — Correlation Matrix

Two views:
1. **Overall correlation** across all team-match observations
2. **Side-by-side** correlation within QF+ vs. Eliminated group — do features relate to each other differently for successful teams?

In [ ]:
CORR_FEATURES = [
    f for f in [
        'network_density', 'degree_centralization', 'top_player_pass_share',
        'clustering_coefficient', 'mean_betweenness_centrality',
        'progressive_pass_share', 'final_third_entry_share',
        'unique_passing_pairs', 'total_completed_passes',
        'lateral_imbalance',
    ]
    if f in mf.columns
]

SHORT_LABELS = {
    'network_density'            : 'Density',
    'degree_centralization'      : 'Centraliz.',
    'top_player_pass_share'      : 'Top Share',
    'clustering_coefficient'     : 'Clustering',
    'mean_betweenness_centrality': 'Btw (mean)',
    'progressive_pass_share'     : 'Prog Pass',
    'final_third_entry_share'    : 'FT Entry',
    'unique_passing_pairs'       : 'Uniq Pairs',
    'total_completed_passes'     : 'Vol (total)',
    'lateral_imbalance'          : 'Lat Imbal.',
}
labels_short = [SHORT_LABELS.get(f, f) for f in CORR_FEATURES]


def corr_heatmap(ax, corr_mat, title, labels):
    n = len(labels)
    im = ax.imshow(corr_mat, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, fontsize=7, rotation=45, ha='right')
    ax.set_yticklabels(labels, fontsize=7)
    for r in range(n):
        for c in range(n):
            val = corr_mat[r, c]
            txt_color = 'white' if abs(val) > 0.65 else 'black'
            ax.text(c, r, f'{val:.2f}', ha='center', va='center',
                    fontsize=6.5, color=txt_color)
    ax.set_title(title, fontsize=10, pad=6)
    return im


corr_all  = mf[CORR_FEATURES].corr().values
corr_qf   = mf[mf['reached_quarterfinal']==1][CORR_FEATURES].corr().values
corr_elim = mf[mf['reached_quarterfinal']==0][CORR_FEATURES].corr().values

fig, axes = plt.subplots(1, 3, figsize=(19, 6.5))

im = corr_heatmap(axes[0], corr_all,  'All teams (n=all match obs)', labels_short)
corr_heatmap(axes[1], corr_qf,   'QF+ teams only',              labels_short)
corr_heatmap(axes[2], corr_elim, 'Eliminated teams only',       labels_short)

plt.colorbar(im, ax=axes, shrink=0.6, label='Pearson r', pad=0.01)
fig.suptitle('Feature Correlation Matrices', fontsize=13)
fig.tight_layout(rect=[0, 0, 0.95, 1])
plt.savefig(FIG_DIR / '04_correlation_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5 — Question-by-Question Analysis

### Q1: Do quarterfinal teams have higher network density?

In [ ]:
feat = 'network_density'
if feat in mf.columns:
    g1 = mf[mf['reached_quarterfinal']==1][feat].dropna()
    g0 = mf[mf['reached_quarterfinal']==0][feat].dropna()
    p, rbc = mwu_row(feat, mf)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # ① Violin
    ax = axes[0]
    parts = ax.violinplot([g0.values, g1.values], positions=[0,1],
                          showmedians=True, showextrema=True)
    for pc, color in zip(parts['bodies'], [ELIM_COLOR, QF_COLOR]):
        pc.set_facecolor(color)
        pc.set_alpha(0.55)
    parts['cmedians'].set_colors(['white', 'white'])
    ax.set_xticks([0,1]); ax.set_xticklabels(['Eliminated','QF+'])
    ax.set_ylabel('Network Density')
    ax.set_title('Violin — Network Density')

    # ② Per-team mean density ranked
    ax = axes[1]
    td = tf[['team','network_density_mean','reached_quarterfinal']].dropna().sort_values(
        'network_density_mean', ascending=True
    )
    colors = [QF_COLOR if q else ELIM_COLOR for q in td['reached_quarterfinal']]
    ax.barh(td['team'], td['network_density_mean'], color=colors, alpha=0.75)
    ax.set_xlabel('Mean Network Density (across matches)')
    ax.set_title('Teams Ranked by Mean Density')
    ax.tick_params(axis='y', labelsize=7)

    # ③ Density per match coloured by group
    ax = axes[2]
    mf_sorted = mf.sort_values(feat)
    colors_m = [QF_COLOR if q else ELIM_COLOR for q in mf_sorted['reached_quarterfinal']]
    ax.scatter(range(len(mf_sorted)), mf_sorted[feat], c=colors_m, alpha=0.4, s=18)
    ax.axhline(g1.mean(), color=QF_COLOR,   lw=2, linestyle='--', label=f'QF+ mean {g1.mean():.3f}')
    ax.axhline(g0.mean(), color=ELIM_COLOR, lw=2, linestyle='--', label=f'Elim mean {g0.mean():.3f}')
    ax.set_xlabel('Match-level observations (sorted)')
    ax.set_ylabel('Network Density')
    ax.set_title('Per-match density (dots)')
    ax.legend(fontsize=8)

    fig.suptitle(
        f'Q1: Network Density — QF+ vs Eliminated  |  r={rbc:+.3f}, p={p:.4f}',
        fontsize=12, y=1.02
    )
    fig.tight_layout()
    plt.savefig(FIG_DIR / '04_Q1_network_density.png', dpi=150, bbox_inches='tight')
    plt.show()

    direction = 'HIGHER' if g1.mean() > g0.mean() else 'LOWER'
    print(f"\n→ QF teams have {direction} network density.")
    print(f"  QF+ mean: {g1.mean():.4f}  |  Elim mean: {g0.mean():.4f}")
    print(f"  Δ = {g1.mean()-g0.mean():+.4f}  |  r={rbc:+.3f}  |  p={p:.4f}")

### Q2: Do quarterfinal teams have lower degree centralization?

In [ ]:
feat = 'degree_centralization'
if feat in mf.columns:
    g1 = mf[mf['reached_quarterfinal']==1][feat].dropna()
    g0 = mf[mf['reached_quarterfinal']==0][feat].dropna()
    p, rbc = mwu_row(feat, mf)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # ① Boxplot with individual match points
    ax = axes[0]
    rng = np.random.default_rng(0)
    for pos, vals, color in [(0, g0.values, ELIM_COLOR), (1, g1.values, QF_COLOR)]:
        bp = ax.boxplot(vals, positions=[pos], widths=0.4, patch_artist=True,
                        boxprops=dict(facecolor=color, alpha=0.4, linewidth=1.3),
                        medianprops=dict(color='black', lw=2.5),
                        whiskerprops=dict(color=color, lw=1.3),
                        capprops=dict(color=color, lw=1.3),
                        flierprops=dict(marker=''))
        jit = rng.uniform(-0.13, 0.13, size=len(vals))
        ax.scatter(pos + jit, vals, color=color, alpha=0.45, s=16, zorder=3)
    ax.set_xticks([0,1]); ax.set_xticklabels(['Eliminated','QF+'], fontsize=10)
    ax.set_ylabel('Degree Centralization (Freeman C)')
    ax.set_title(f'Degree Centralization\nr={rbc:+.3f}  p={p:.4f}')

    # ② Team-level bar sorted by centralization
    ax = axes[1]
    col = 'degree_centralization_mean'
    if col in tf.columns:
        td = tf[['team', col, 'reached_quarterfinal']].dropna().sort_values(col)
        colors = [ELIM_COLOR if q == 0 else QF_COLOR for q in td['reached_quarterfinal']]
        bars = ax.barh(td['team'], td[col], color=colors, alpha=0.75)
        ax.set_xlabel('Mean Degree Centralization')
        ax.set_title('Teams Ranked by Mean Centralization')
        ax.tick_params(axis='y', labelsize=7)
        handles = [
            mpatches.Patch(facecolor=QF_COLOR,   alpha=0.7, label='QF+'),
            mpatches.Patch(facecolor=ELIM_COLOR, alpha=0.7, label='Eliminated'),
        ]
        ax.legend(handles=handles, fontsize=8)

    fig.suptitle('Q2: Do QF Teams Have Lower Degree Centralization?', fontsize=12)
    fig.tight_layout()
    plt.savefig(FIG_DIR / '04_Q2_centralization.png', dpi=150, bbox_inches='tight')
    plt.show()

    direction = 'LOWER' if g1.mean() < g0.mean() else 'HIGHER'
    print(f"\n→ QF teams have {direction} degree centralization.")
    print(f"  QF+ mean: {g1.mean():.4f}  |  Elim mean: {g0.mean():.4f}")
    print(f"  Δ = {g1.mean()-g0.mean():+.4f}  |  r={rbc:+.3f}  |  p={p:.4f}")

### Q3: Do quarterfinal teams have more unique passing pairs?

In [ ]:
feat = 'unique_passing_pairs'
if feat in mf.columns:
    g1 = mf[mf['reached_quarterfinal']==1][feat].dropna()
    g0 = mf[mf['reached_quarterfinal']==0][feat].dropna()
    p, rbc = mwu_row(feat, mf)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # ① Histogram overlay
    ax = axes[0]
    bins = np.linspace(
        min(g0.min(), g1.min()), max(g0.max(), g1.max()), 25
    )
    ax.hist(g0, bins=bins, color=ELIM_COLOR, alpha=0.55, label='Eliminated', density=True)
    ax.hist(g1, bins=bins, color=QF_COLOR,   alpha=0.55, label='QF+',        density=True)
    ax.axvline(g0.mean(), color=ELIM_COLOR, lw=2, linestyle=':', label=f'Elim mean {g0.mean():.0f}')
    ax.axvline(g1.mean(), color=QF_COLOR,   lw=2, linestyle=':', label=f'QF+ mean {g1.mean():.0f}')
    ax.set_xlabel('Unique Passing Pairs (directed edges in graph)')
    ax.set_ylabel('Density')
    ax.set_title(f'Unique Passing Pairs Distribution\nr={rbc:+.3f}  p={p:.4f}')
    ax.legend(fontsize=8)

    # ② Passing pairs vs total passes scatter
    ax = axes[1]
    for group_val, color, label in [(0, ELIM_COLOR, 'Eliminated'), (1, QF_COLOR, 'QF+')]:
        sub = mf[mf['reached_quarterfinal'] == group_val]
        ax.scatter(sub['total_completed_passes'], sub[feat],
                   c=color, alpha=0.4, s=20, label=label)
    ax.set_xlabel('Total Completed Passes')
    ax.set_ylabel('Unique Passing Pairs')
    ax.set_title('Diversity vs. Volume\n(higher = more player combinations used)')
    ax.legend(fontsize=8)

    fig.suptitle('Q3: Do QF Teams Use More Unique Passing Pairs?', fontsize=12)
    fig.tight_layout()
    plt.savefig(FIG_DIR / '04_Q3_unique_pairs.png', dpi=150, bbox_inches='tight')
    plt.show()

    direction = 'MORE' if g1.mean() > g0.mean() else 'FEWER'
    print(f"\n→ QF teams use {direction} unique passing pairs per match.")
    print(f"  QF+ mean: {g1.mean():.1f}  |  Elim mean: {g0.mean():.1f}")
    print(f"  Δ = {g1.mean()-g0.mean():+.1f}  |  r={rbc:+.3f}  |  p={p:.4f}")

### Q4: Do quarterfinal teams have higher progressive pass share?

In [ ]:
feat = 'progressive_pass_share'
if feat in mf.columns:
    g1 = mf[mf['reached_quarterfinal']==1][feat].dropna()
    g0 = mf[mf['reached_quarterfinal']==0][feat].dropna()
    p, rbc = mwu_row(feat, mf)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # ① KDE overlay
    ax = axes[0]
    for vals, color, label in [(g0.values, ELIM_COLOR, 'Eliminated'),
                                (g1.values, QF_COLOR,   'QF+')]:
        x_grid = np.linspace(vals.min()-0.02, vals.max()+0.02, 300)
        kde = gaussian_kde(vals, bw_method='scott')
        ax.fill_between(x_grid, kde(x_grid), alpha=0.25, color=color)
        ax.plot(x_grid, kde(x_grid), color=color, lw=2, label=label)
        ax.axvline(vals.mean(), color=color, lw=1.5, linestyle='--')
    ax.set_xlabel('Progressive Pass Share')
    ax.set_ylabel('Density')
    ax.set_title(f'KDE — Progressive Pass Share\nr={rbc:+.3f}  p={p:.4f}')
    ax.legend(fontsize=8)

    # ② Boxplot stage breakdown
    if 'stage_label' in mf.columns:
        ax = axes[1]
        stage_order = ['group', 'R16', 'QF', 'SF', '3rd', 'Final']
        stage_order = [s for s in stage_order if s in mf['stage_label'].values]
        bp_data = [mf[mf['stage_label']==s][feat].dropna().values for s in stage_order]
        bp = ax.boxplot(bp_data, positions=range(len(stage_order)), patch_artist=True,
                        medianprops=dict(color='black', lw=2))
        stage_colors = [NEUTRAL, NEUTRAL, QF_COLOR, QF_COLOR, QF_COLOR, QF_COLOR]
        for patch, color in zip(bp['boxes'], stage_colors[:len(stage_order)]):
            patch.set_facecolor(color)
            patch.set_alpha(0.5)
        ax.set_xticks(range(len(stage_order)))
        ax.set_xticklabels(stage_order, fontsize=8)
        ax.set_ylabel('Progressive Pass Share')
        ax.set_title('Progressive Pass Share by Stage')

    # ③ Team-level bar chart
    ax = axes[2]
    col = 'progressive_pass_share_mean'
    if col in tf.columns:
        td = tf[['team', col, 'reached_quarterfinal']].dropna().sort_values(col)
        colors = [QF_COLOR if q else ELIM_COLOR for q in td['reached_quarterfinal']]
        ax.barh(td['team'], td[col], color=colors, alpha=0.75)
        ax.set_xlabel('Mean Progressive Pass Share')
        ax.set_title('Teams Ranked by Prog. Pass Share')
        ax.tick_params(axis='y', labelsize=7)

    fig.suptitle('Q4: Do QF Teams Have Higher Progressive Pass Share?', fontsize=12)
    fig.tight_layout()
    plt.savefig(FIG_DIR / '04_Q4_progressive_passes.png', dpi=150, bbox_inches='tight')
    plt.show()

    direction = 'HIGHER' if g1.mean() > g0.mean() else 'LOWER'
    print(f"\n→ QF teams have {direction} progressive pass share.")
    print(f"  QF+ mean: {g1.mean():.4f}  |  Elim mean: {g0.mean():.4f}")
    print(f"  Δ = {g1.mean()-g0.mean():+.4f}  |  r={rbc:+.3f}  |  p={p:.4f}")

### Q5: Are successful teams more balanced or more reliant on a single hub?

In [ ]:
hub_feats = [
    ('top_player_pass_share',      'Top Player Pass Share'),
    ('degree_centralization',      'Degree Centralization'),
    ('mean_betweenness_centrality','Mean Betweenness'),
]
hub_feats = [(f, l) for f, l in hub_feats if f in mf.columns]

fig, axes = plt.subplots(1, len(hub_feats) + 1, figsize=(5 * (len(hub_feats)+1), 5))
rng = np.random.default_rng(7)

for i, (feat, feat_label) in enumerate(hub_feats):
    ax = axes[i]
    g1 = mf[mf['reached_quarterfinal']==1][feat].dropna().values
    g0 = mf[mf['reached_quarterfinal']==0][feat].dropna().values
    p, rbc = mwu_row(feat, mf)

    for pos, vals, color in [(0, g0, ELIM_COLOR), (1, g1, QF_COLOR)]:
        bp = ax.boxplot(vals, positions=[pos], widths=0.4, patch_artist=True,
                        boxprops=dict(facecolor=color, alpha=0.45, lw=1.2),
                        medianprops=dict(color='black', lw=2.5),
                        whiskerprops=dict(color=color), capprops=dict(color=color),
                        flierprops=dict(marker=''))
        jit = rng.uniform(-0.13, 0.13, size=len(vals))
        ax.scatter(pos + jit, vals, color=color, alpha=0.4, s=16, zorder=3)

    ax.set_xticks([0,1]); ax.set_xticklabels(['Elim','QF+'], fontsize=9)
    ax.set_title(f'{feat_label}\nr={rbc:+.3f}  p={p:.4f}', fontsize=9)
    ax.set_ylabel(feat_label, fontsize=8)

# ④ Scatter: centralization vs density (key tradeoff)
ax = axes[-1]
for group_val, color, label in [(0, ELIM_COLOR, 'Eliminated'), (1, QF_COLOR, 'QF+')]:
    sub = mf[mf['reached_quarterfinal']==group_val].dropna(
        subset=['network_density','degree_centralization']
    )
    ax.scatter(sub['network_density'], sub['degree_centralization'],
               c=color, alpha=0.4, s=22, label=label)
ax.set_xlabel('Network Density (higher = more connected)')
ax.set_ylabel('Degree Centralization (higher = more hub-dependent)')
ax.set_title('Density vs. Centralization\n(ideal QF profile: top-right = dense + balanced?)')
ax.legend(fontsize=8)

# Add quadrant shading
ax.axhline(mf['degree_centralization'].median(), color=NEUTRAL, lw=0.8, linestyle=':')
ax.axvline(mf['network_density'].median(),       color=NEUTRAL, lw=0.8, linestyle=':')
ax.text(mf['network_density'].quantile(0.85), mf['degree_centralization'].quantile(0.1),
        'Dense &\nBalanced', fontsize=7, color='green', alpha=0.7, ha='center')
ax.text(mf['network_density'].quantile(0.15), mf['degree_centralization'].quantile(0.85),
        'Sparse &\nHub-driven', fontsize=7, color='red', alpha=0.7, ha='center')

fig.suptitle('Q5: Hub Reliance vs. Balanced Passing — Which Profile Advances?',
             fontsize=12, y=1.02)
fig.tight_layout()
plt.savefig(FIG_DIR / '04_Q5_hub_vs_balanced.png', dpi=150, bbox_inches='tight')
plt.show()

for feat, feat_label in hub_feats:
    g1 = mf[mf['reached_quarterfinal']==1][feat].dropna()
    g0 = mf[mf['reached_quarterfinal']==0][feat].dropna()
    p, rbc = mwu_row(feat, mf)
    direction = 'LOWER' if g1.mean() < g0.mean() else 'HIGHER'
    print(f"  {feat_label:30s}: QF+ is {direction} ({g1.mean():.4f} vs {g0.mean():.4f})"
          f"  r={rbc:+.3f}  p={p:.4f}")

---
## Section 6 — Feature Comparison Chart (Effect Size Summary)

A single chart ranking all features by the magnitude and direction of the difference between QF and eliminated teams.  
This is the most efficient single-page summary of the full exploratory analysis.

In [ ]:
all_feats = [
    f for f in [
        'network_density', 'degree_centralization', 'top_player_pass_share',
        'unique_passing_pairs', 'total_completed_passes', 'average_weighted_degree',
        'clustering_coefficient', 'mean_betweenness_centrality',
        'max_betweenness_centrality', 'progressive_pass_share',
        'final_third_entry_share', 'centre_channel_share',
        'lateral_imbalance', 'number_of_players_in_network',
    ]
    if f in mf.columns
]

effect_rows = []
for feat in all_feats:
    g1 = mf[mf['reached_quarterfinal']==1][feat].dropna()
    g0 = mf[mf['reached_quarterfinal']==0][feat].dropna()
    p, rbc = mwu_row(feat, mf)
    effect_rows.append({
        'feature': LABELS.get(feat, feat),
        'effect_size': rbc,
        'p_value': p,
        'significant': p < 0.05 if p is not np.nan else False,
        'direction': 'QF+ higher' if (g1.mean() > g0.mean()) else 'QF+ lower',
    })

eff_df = pd.DataFrame(effect_rows).sort_values('effect_size', ascending=True)
eff_df.to_csv(TAB_DIR / '04_effect_sizes.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 6))

colors = [
    QF_COLOR if (sig and eff > 0)
    else ELIM_COLOR if (sig and eff < 0)
    else '#aaaaaa'
    for sig, eff in zip(eff_df['significant'], eff_df['effect_size'])
]

bars = ax.barh(eff_df['feature'], eff_df['effect_size'],
               color=colors, alpha=0.8, edgecolor='white', linewidth=0.5)

# Annotate p-values on significant bars
for bar, (_, row) in zip(bars, eff_df.iterrows()):
    if row['significant']:
        x = bar.get_width()
        ax.text(x + 0.005 * np.sign(x), bar.get_y() + bar.get_height()/2,
                f"p={row['p_value']:.3f}", va='center',
                fontsize=7, color='black')

ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Effect Size: rank-biserial correlation r\n'
              '(positive = QF+ teams are higher; negative = QF+ teams are lower)',
              fontsize=9)
ax.set_title('Feature Comparison: Quarterfinal vs. Eliminated Teams\n'
             'Coloured bars are statistically significant (p < 0.05)',
             fontsize=11)

legend_handles = [
    mpatches.Patch(facecolor=QF_COLOR,   alpha=0.8, label='Significant — QF+ teams higher'),
    mpatches.Patch(facecolor=ELIM_COLOR, alpha=0.8, label='Significant — Eliminated teams higher'),
    mpatches.Patch(facecolor='#aaaaaa',  alpha=0.8, label='Not significant (p ≥ 0.05)'),
]
ax.legend(handles=legend_handles, fontsize=8, loc='lower right')
ax.set_xlim(-0.55, 0.55)
ax.set_facecolor('#f9f9f9')
fig.tight_layout()
plt.savefig(FIG_DIR / '04_effect_size_summary.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7 — Team-Level Heatmap (All 32 Teams × All Features)

A single overview table showing where every team sits on each feature. Teams are grouped and sorted by advancement stage.

In [ ]:
team_feat_cols = [
    c for c in [
        'network_density_mean', 'degree_centralization_mean',
        'top_player_pass_share_mean', 'clustering_coefficient_mean',
        'mean_betweenness_centrality_mean', 'progressive_pass_share_mean',
        'final_third_entry_share_mean', 'lateral_imbalance_mean',
        'centre_channel_share_mean',
    ]
    if c in tf.columns
]

short_col_labels = [
    c.replace('_mean','').replace('_',' ').replace('mean ','').title()
    for c in team_feat_cols
]

# Sort: QF teams first, then alphabetically within each group
tf_sorted = tf.sort_values(
    ['reached_quarterfinal','team'], ascending=[False, True]
).reset_index(drop=True)

# Z-score each column for colour scaling
mat = tf_sorted[team_feat_cols].copy()
mat_z = (mat - mat.mean()) / mat.std()

fig, ax = plt.subplots(figsize=(len(team_feat_cols)*1.4 + 2, len(tf_sorted)*0.38 + 2))
im = ax.imshow(mat_z.values, cmap='RdYlGn', aspect='auto', vmin=-2, vmax=2)

ax.set_xticks(range(len(team_feat_cols)))
ax.set_xticklabels(short_col_labels, fontsize=8, rotation=40, ha='right')
ax.set_yticks(range(len(tf_sorted)))

y_labels = [
    f"{'★ ' if q else '  '}{team}"
    for team, q in zip(tf_sorted['team'], tf_sorted['reached_quarterfinal'])
]
ax.set_yticklabels(y_labels, fontsize=8)

# Divider line between QF and eliminated groups
n_qf = (tf_sorted['reached_quarterfinal'] == 1).sum()
ax.axhline(n_qf - 0.5, color='black', lw=2)

plt.colorbar(im, ax=ax, shrink=0.5, label='Z-score (green = above mean, red = below mean)')
ax.set_title(
    'Team Feature Heatmap (★ = reached quarterfinal)\n'
    'Z-scored — green means above average for that feature',
    fontsize=11, pad=10
)
fig.tight_layout()
plt.savefig(FIG_DIR / '04_team_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 8 — EDA Conclusions

Answers to the five research questions, drawn from the analysis above.

In [ ]:
conclusions = {}

for feat, question in [
    ('network_density',        'Q1 — Network Density'),
    ('degree_centralization',  'Q2 — Degree Centralization'),
    ('unique_passing_pairs',   'Q3 — Unique Passing Pairs'),
    ('progressive_pass_share', 'Q4 — Progressive Pass Share'),
    ('top_player_pass_share',  'Q5 — Hub Reliance (Top Player Share)'),
]:
    if feat not in mf.columns:
        conclusions[question] = '(feature not available)'
        continue
    g1 = mf[mf['reached_quarterfinal']==1][feat].dropna()
    g0 = mf[mf['reached_quarterfinal']==0][feat].dropna()
    p, rbc = mwu_row(feat, mf)
    delta = g1.mean() - g0.mean()
    sig_str = 'statistically significant' if p < 0.05 else 'not statistically significant'
    dir_str = 'HIGHER' if delta > 0 else 'LOWER'
    conclusions[question] = (
        f"QF+ teams: {dir_str} by {abs(delta):.4f}  "
        f"(r={rbc:+.3f}, p={p:.4f}) — {sig_str}"
    )

print("=" * 68)
print("EDA CONCLUSIONS")
print("=" * 68)
for q, answer in conclusions.items():
    print(f"\n{q}")
    print(f"  {answer}")

print("\n" + "=" * 68)
print("OVERALL INTERPRETATION")
print("=" * 68)
print("""
With n=32 teams the sample is small, so individual p-values should be
interpreted cautiously. Effect sizes (rank-biserial r) are the more
informative metric at this scale.

The pattern that emerges is that quarterfinal teams tend to exhibit:
  • More connected networks (higher density)
  • More distributed passing (lower centralization, lower hub reliance)
  • More diverse player combinations (more unique passing pairs)
  • More forward-oriented play (higher progressive pass share)

This is consistent with the hypothesis that successful teams rely on
collective, balanced passing structures rather than routing everything
through one dominant player.

These patterns carry forward to Notebook 03 for formal modeling.
""")